# 📖 Notebook 2: Consistency vs Availability Demo

Now let's get hands-on. We have a PostgreSQL **primary** (for writes) and a **replica** (for reads). We'll observe replication lag, simulate partitions, and compare CP and AP behavior.

## Learning Objectives

By the end of this notebook, you'll understand:
- How PostgreSQL streaming replication works
- What replication lag looks like in practice
- How CP systems behave during a partition (reject stale reads)
- How AP systems behave during a partition (serve stale data from cache)
- The concept of "read-your-own-writes" consistency

## 🛠️ Setup

Start the infrastructure first:

```bash
cd core-concepts/cap-theorem
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import time
import json
import subprocess

PRIMARY_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "cap_demo",
    "user": "demo",
    "password": "demo"
}

REPLICA_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "database": "cap_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_primary():
    return psycopg2.connect(**PRIMARY_CONFIG)

def get_replica():
    return psycopg2.connect(**REPLICA_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Verify connections
for name, fn in [("Primary", get_primary), ("Replica", get_replica)]:
    try:
        c = fn(); c.close()
        print(f"✅ {name} connected")
    except Exception as e:
        print(f"❌ {name} failed: {e}")

try:
    r = get_redis(); r.ping()
    print("✅ Redis connected")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 📡 How Replication Works

Our lab uses PostgreSQL **streaming replication**:

```
┌────────────────┐         WAL stream         ┌────────────────┐
│   PRIMARY      │ ───────────────────────────▶│   REPLICA      │
│   (port 5432)  │                             │   (port 5433)  │
│                │  Every write on the primary  │                │
│  ✅ Reads      │  is shipped to the replica   │  ✅ Reads      │
│  ✅ Writes     │  via WAL (Write-Ahead Log)   │  ❌ Writes     │
│                │                             │  (read-only)   │
└────────────────┘                             └────────────────┘
```

**WAL** (Write-Ahead Log): Every change is first written to a log file, then the log is streamed to the replica. The replica replays these changes to stay in sync.

This is **asynchronous replication** by default — the primary doesn't wait for the replica to confirm. This means the replica can fall slightly behind.

In [ ]:
# Let's check the replication status on the primary

conn = get_primary()
cursor = conn.cursor()

# Check if replication is active
cursor.execute("""
    SELECT
        client_addr,
        state,
        sent_lsn,
        replay_lsn,
        pg_wal_lsn_diff(sent_lsn, replay_lsn) AS lag_bytes
    FROM pg_stat_replication
""")

rows = cursor.fetchall()
conn.close()

if rows:
    print("📡 Active Replication Connections:")
    print("=" * 60)
    for row in rows:
        print(f"   Client:     {row[0]}")
        print(f"   State:      {row[1]}")
        print(f"   Sent LSN:   {row[2]}")
        print(f"   Replay LSN: {row[3]}")
        print(f"   Lag:        {row[4]} bytes")
    print()
    print("✅ Streaming replication is active!")
else:
    print("❌ No replication connections found.")
    print("   Make sure the replica is running: docker-compose up -d")

## ⏱️ Measuring Replication Lag

Even without a partition, there's always a small delay between writing to the primary and the data appearing on the replica. Let's measure it.

In [ ]:
# Measure replication lag: write to primary, poll replica until it catches up

def measure_replication_lag(update_sql, check_sql, expected_value):
    """Write to primary, measure how long until replica sees the change."""
    # Write to primary
    conn = get_primary()
    conn.autocommit = True
    cursor = conn.cursor()
    cursor.execute(update_sql)
    conn.close()
    write_time = time.time()

    # Poll the replica
    max_wait = 5.0  # seconds
    poll_interval = 0.001  # 1ms
    elapsed = 0

    while elapsed < max_wait:
        try:
            conn = get_replica()
            cursor = conn.cursor()
            cursor.execute(check_sql)
            result = cursor.fetchone()[0]
            conn.close()

            if str(result) == str(expected_value):
                return (time.time() - write_time) * 1000  # ms
        except:
            pass

        time.sleep(poll_interval)
        elapsed = time.time() - write_time

    return -1  # timeout

# Run multiple replication lag measurements
print("⏱️  Measuring Replication Lag (10 writes)")
print("=" * 50)
print()

lags = []
for i in range(10):
    new_name = f"Lag Test {i}"
    lag = measure_replication_lag(
        f"UPDATE user_profiles SET display_name = '{new_name}' WHERE id = 2",
        "SELECT display_name FROM user_profiles WHERE id = 2",
        new_name
    )
    lags.append(lag)
    status = f"{lag:.1f} ms" if lag >= 0 else "TIMEOUT"
    print(f"   Write {i+1:>2}: {status}")

valid_lags = [l for l in lags if l >= 0]
if valid_lags:
    print()
    print(f"📊 Results:")
    print(f"   Average lag: {sum(valid_lags)/len(valid_lags):.1f} ms")
    print(f"   Min lag:     {min(valid_lags):.1f} ms")
    print(f"   Max lag:     {max(valid_lags):.1f} ms")
    print()
    print("💡 Replication lag is usually under 10ms on a local network.")
    print("   In production across data centers, it can be 50-500ms or more.")

# Reset
conn = get_primary()
conn.autocommit = True
cursor = conn.cursor()
cursor.execute("UPDATE user_profiles SET display_name = 'Bob Smith' WHERE id = 2")
conn.close()

## 🔌 Simulating a Network Partition

Now for the exciting part! We'll **pause the replica** using `docker pause` to simulate a network partition. The primary keeps accepting writes, but the replica is frozen — it can't receive updates.

```
┌────────────────┐         ✗ PAUSED ✗         ┌────────────────┐
│   PRIMARY      │ ─── ── ── ── ── ── ── ───▶│   REPLICA      │
│   (port 5432)  │    (WAL stream broken)     │   (port 5433)  │
│                │                             │   ⏸️ FROZEN     │
│  Keeps writing │                             │  Stale data    │
└────────────────┘                             └────────────────┘
```

This simulates what happens when servers in different data centers lose connectivity.

In [ ]:
# Step 1: Read the current state from both servers

def read_profile_from(source_name, connect_fn):
    """Read Alice's profile from a given database."""
    try:
        conn = connect_fn()
        cursor = conn.cursor()
        cursor.execute("SELECT display_name, bio, profile_views FROM user_profiles WHERE username = 'alice'")
        row = cursor.fetchone()
        conn.close()
        return {"name": row[0], "bio": row[1], "views": row[2]}
    except Exception as e:
        return {"error": str(e)}

print("📸 State BEFORE partition:")
print()
primary_data = read_profile_from("Primary", get_primary)
replica_data = read_profile_from("Replica", get_replica)
print(f"   Primary: {primary_data}")
print(f"   Replica: {replica_data}")
print()
if primary_data == replica_data:
    print("✅ Both servers in sync")
else:
    print("⚠️  Already out of sync!")

In [ ]:
# Step 2: PAUSE the replica (simulate partition)

print("🔌 Simulating network partition...")
print()
result = subprocess.run(["docker", "pause", "cap-postgres-replica"], capture_output=True, text=True)
if result.returncode == 0:
    print("⏸️  Replica PAUSED — it's now frozen and can't receive updates")
    print("   This simulates a network partition between primary and replica")
else:
    print(f"❌ Failed to pause: {result.stderr}")

In [ ]:
# Step 3: Write to primary WHILE the replica is paused

conn = get_primary()
conn.autocommit = True
cursor = conn.cursor()
cursor.execute("""
    UPDATE user_profiles
    SET display_name = 'Alice (Name Changed During Partition!)',
        bio = 'This bio was updated while the replica was frozen',
        profile_views = profile_views + 999,
        updated_at = NOW()
    WHERE username = 'alice'
""")
conn.close()

print("✏️  Updated Alice's profile on PRIMARY:")
print("   Name: 'Alice (Name Changed During Partition!)'")
print("   Bio:  'This bio was updated while the replica was frozen'")
print("   Views: +999")
print()
print("The replica is paused — it knows NOTHING about this update.")

In [ ]:
# Step 4: Compare — primary has new data, replica has old data

print("📸 State DURING partition:")
print("=" * 65)
print()

primary_data = read_profile_from("Primary", get_primary)
print(f"   PRIMARY (has the latest write):")
print(f"     Name:  {primary_data.get('name', 'ERROR')}")
print(f"     Bio:   {primary_data.get('bio', 'ERROR')}")
print(f"     Views: {primary_data.get('views', 'ERROR')}")
print()

# The replica is paused, so we can't connect to it.
# This is exactly what happens during a real partition!
print(f"   REPLICA (frozen / partitioned):")
replica_data = read_profile_from("Replica", get_replica)
if "error" in replica_data:
    print(f"     ❌ Cannot connect: replica is unreachable")
    print(f"     This is what a partition looks like!")
else:
    print(f"     Name:  {replica_data.get('name', '???')}")
    print(f"     Bio:   {replica_data.get('bio', '???')}")
    print(f"     Views: {replica_data.get('views', '???')}")
    print(f"     ⚠️  This is STALE data!")

print()
print("🤔 What should our application do right now?")
print("   CP approach: Return an error — 'Cannot guarantee data freshness'")
print("   AP approach: Return stale data from cache — 'Here's what we last knew'")

In [ ]:
# Step 5: Demonstrate CP vs AP application behavior

# First, let's pre-populate a cache snapshot (like a real AP system would have)
r = get_redis()
cached_profile = {
    "name": "Alice Johnson",
    "bio": "Software engineer who loves distributed systems",
    "views": 1500,
    "cached_at": "before partition"
}
r.set("user:alice:profile", json.dumps(cached_profile))

def cp_read_profile():
    """CP approach: only return data if we can verify it's fresh."""
    try:
        conn = get_replica()
        cursor = conn.cursor()
        cursor.execute("SELECT display_name FROM user_profiles WHERE username = 'alice'")
        row = cursor.fetchone()
        conn.close()
        return {"status": "ok", "data": row[0]}
    except Exception:
        # Can't reach replica — refuse to serve potentially stale data
        return {"status": "error", "message": "Service unavailable — cannot guarantee consistency"}

def ap_read_profile():
    """AP approach: always return data, even if stale."""
    try:
        conn = get_replica()
        cursor = conn.cursor()
        cursor.execute("SELECT display_name FROM user_profiles WHERE username = 'alice'")
        row = cursor.fetchone()
        conn.close()
        return {"status": "ok", "data": row[0], "source": "replica"}
    except Exception:
        # Replica down? Fall back to cache — stale but available
        cached = r.get("user:alice:profile")
        if cached:
            data = json.loads(cached)
            return {"status": "ok", "data": data["name"], "source": "cache (stale)", "warning": "Data may be outdated"}
        return {"status": "error", "message": "No data available"}

print("🔒 CP Approach (Consistency Priority):")
result = cp_read_profile()
if result["status"] == "error":
    print(f"   ❌ {result['message']}")
    print(f"   → User sees an error page. Frustrating, but data integrity is preserved.")
else:
    print(f"   ✅ Got data: {result['data']}")

print()
print("🌐 AP Approach (Availability Priority):")
result = ap_read_profile()
print(f"   ✅ Got data: {result['data']}")
print(f"   Source: {result.get('source', 'unknown')}")
if "warning" in result:
    print(f"   ⚠️  {result['warning']}")
print(f"   → User sees a page. Data might be stale, but the app works.")

print()
print("💡 Neither approach is 'better' — it depends on your use case!")
print("   Bank balance → CP (wrong balance is dangerous)")
print("   Profile page → AP (stale name is fine)")

In [ ]:
# Step 6: UNPAUSE the replica (heal the partition)

print("🔌 Healing the partition...")
result = subprocess.run(["docker", "unpause", "cap-postgres-replica"], capture_output=True, text=True)
if result.returncode == 0:
    print("▶️  Replica UNPAUSED — it will now catch up with the primary")
else:
    print(f"❌ Failed to unpause: {result.stderr}")

# Wait for replication to catch up
print("   Waiting for replica to catch up...")
time.sleep(2)

# Check if they're in sync now
primary_data = read_profile_from("Primary", get_primary)
replica_data = read_profile_from("Replica", get_replica)

print()
print("📸 State AFTER healing partition:")
print(f"   Primary: name = {primary_data.get('name', 'ERROR')}")
print(f"   Replica: name = {replica_data.get('name', 'ERROR')}")
print()

if primary_data.get("name") == replica_data.get("name"):
    print("✅ Replica caught up! Both servers are consistent again.")
    print("   This is 'eventual consistency' in action — the system healed itself.")
else:
    print("⏳ Replica still catching up... try running this cell again.")

## 📖 Read-Your-Own-Writes Consistency

A common middle ground: after a user writes, make sure **that same user** sees their own update immediately — even if other users still see stale data.

```
Alice updates her bio:
  ✅ Alice reads → sees new bio (read from PRIMARY)
  ⏳ Bob reads   → sees old bio (read from REPLICA, not yet replicated)
  ⏳ Carol reads → sees old bio (read from REPLICA, not yet replicated)
  ...eventually, replica catches up...
  ✅ Bob reads   → sees new bio
  ✅ Carol reads → sees new bio
```

This is used by many social media platforms — you always see your own changes instantly.

In [ ]:
# Implementing read-your-own-writes
# Strategy: after a write, read from primary for THAT user, replica for everyone else

# Track which users have recent writes (in a real system, use Redis with TTL)
recent_writers = {}  # user_id → timestamp of last write
WRITER_READ_PRIMARY_WINDOW = 5  # seconds to read from primary after a write

def write_profile(user_id, new_name):
    """Write a profile update to the primary."""
    conn = get_primary()
    conn.autocommit = True
    cursor = conn.cursor()
    cursor.execute("UPDATE user_profiles SET display_name = %s, updated_at = NOW() WHERE id = %s", (new_name, user_id))
    conn.close()
    recent_writers[user_id] = time.time()
    return new_name

def read_profile(user_id, reader_id):
    """Read-your-own-writes: writers read from primary, others from replica."""
    # If the reader recently wrote, route to primary for freshness
    last_write = recent_writers.get(reader_id, 0)
    use_primary = (time.time() - last_write) < WRITER_READ_PRIMARY_WINDOW

    source = "PRIMARY" if use_primary else "REPLICA"
    connect_fn = get_primary if use_primary else get_replica

    conn = connect_fn()
    cursor = conn.cursor()
    cursor.execute("SELECT display_name FROM user_profiles WHERE id = %s", (user_id,))
    name = cursor.fetchone()[0]
    conn.close()
    return name, source

# Demo: Alice (user 1) updates her profile
print("📖 Read-Your-Own-Writes Demo")
print("=" * 55)
print()

# Alice updates her own profile
write_profile(1, "Alice (Just Updated!)")
print("✏️  Alice (user 1) updated her name to 'Alice (Just Updated!)'")
print()

# Alice reads her own profile → routed to PRIMARY
name, source = read_profile(user_id=1, reader_id=1)
print(f"👤 Alice reads her OWN profile:")
print(f"   Name:   {name}")
print(f"   Source: {source}")
print(f"   ✅ She sees her own update immediately!")
print()

# Bob reads Alice's profile → routed to REPLICA
name, source = read_profile(user_id=1, reader_id=2)
print(f"👤 Bob reads Alice's profile:")
print(f"   Name:   {name}")
print(f"   Source: {source}")
if "Updated" not in name:
    print(f"   ⏳ Bob sees the old name (replica hasn't caught up yet)")
else:
    print(f"   ✅ Replica already caught up (fast replication!)")

print()
print("💡 Read-your-own-writes is a great middle ground:")
print("   - Writers get instant feedback (reads from primary)")
print("   - Everyone else gets fast reads (from replica)")
print("   - No one notices the brief inconsistency")

## 🏦 Consistency Matters: Bank Transfer Demo

Let's see why financial systems MUST use CP. We'll simulate a bank transfer and show what goes wrong with AP.

In [ ]:
# CP-style bank transfer: use transactions on the PRIMARY to guarantee consistency

def transfer_money_cp(from_account, to_account, amount):
    """Transfer money using a database transaction (CP approach)."""
    conn = get_primary()
    try:
        cursor = conn.cursor()

        # Check balance first
        cursor.execute("SELECT balance FROM bank_accounts WHERE id = %s FOR UPDATE", (from_account,))
        balance = float(cursor.fetchone()[0])

        if balance < amount:
            conn.rollback()
            return False, f"Insufficient funds: ${balance:.2f} < ${amount:.2f}"

        # Debit and credit
        cursor.execute("UPDATE bank_accounts SET balance = balance - %s WHERE id = %s", (amount, from_account))
        cursor.execute("UPDATE bank_accounts SET balance = balance + %s WHERE id = %s", (amount, to_account))

        # Log the transaction
        cursor.execute(
            "INSERT INTO transactions (from_account, to_account, amount) VALUES (%s, %s, %s)",
            (from_account, to_account, amount)
        )

        conn.commit()
        return True, f"Transferred ${amount:.2f}"
    except Exception as e:
        conn.rollback()
        return False, str(e)
    finally:
        conn.close()

# Show initial balances
conn = get_primary()
cursor = conn.cursor()
cursor.execute("SELECT id, account_name, balance FROM bank_accounts ORDER BY id")
print("🏦 Bank Accounts (Before Transfer):")
print(f"   {'ID':>3}  {'Account':<20}  {'Balance':>10}")
print("   " + "-" * 40)
for row in cursor.fetchall():
    print(f"   {row[0]:>3}  {row[1]:<20}  ${float(row[2]):>9.2f}")
conn.close()

# Transfer $1000 from Alice Checking to Bob Checking
print()
success, msg = transfer_money_cp(from_account=1, to_account=3, amount=1000)
print(f"💸 Transfer: Alice Checking → Bob Checking: $1000")
print(f"   Result: {'✅' if success else '❌'} {msg}")

# Show updated balances
print()
conn = get_primary()
cursor = conn.cursor()
cursor.execute("SELECT id, account_name, balance FROM bank_accounts ORDER BY id")
print("🏦 Bank Accounts (After Transfer):")
print(f"   {'ID':>3}  {'Account':<20}  {'Balance':>10}")
print("   " + "-" * 40)
for row in cursor.fetchall():
    print(f"   {row[0]:>3}  {row[1]:<20}  ${float(row[2]):>9.2f}")
conn.close()

print()
print("💡 Notice: both accounts changed atomically inside a transaction.")
print("   The money was never 'lost' or 'duplicated'. That's CP consistency!")
print("   SELECT ... FOR UPDATE locks the row so no one else can read a stale balance.")

## 🧹 Cleanup

In [ ]:
# Reset all data we modified
conn = get_primary()
conn.autocommit = True
cursor = conn.cursor()

# Reset profiles
cursor.execute("UPDATE user_profiles SET display_name = 'Alice Johnson', bio = 'Software engineer who loves distributed systems', updated_at = NOW() WHERE username = 'alice'")
cursor.execute("UPDATE user_profiles SET display_name = 'Bob Smith' WHERE id = 2")

# Reset bank accounts
cursor.execute("UPDATE bank_accounts SET balance = 5000.00 WHERE id = 1")
cursor.execute("UPDATE bank_accounts SET balance = 3000.00 WHERE id = 3")
cursor.execute("DELETE FROM transactions")
cursor.execute("DELETE FROM seat_reservations")

conn.close()

# Clean Redis
r = get_redis()
for pattern in ["user:*", "post:*"]:
    keys = r.keys(pattern)
    if keys:
        r.delete(*keys)

# Make sure replica is running
subprocess.run(["docker", "unpause", "cap-postgres-replica"], capture_output=True)

print("🧹 Cleaned up all demo data")
print("🧹 Ensured replica is running")

## 📚 Summary

### Key Takeaways

1. **Replication lag** is real — even healthy replicas are slightly behind the primary
2. **During a partition** (replica frozen), the primary and replica diverge
3. **CP approach**: refuse to serve data if you can't guarantee freshness (errors during partition)
4. **AP approach**: serve data from cache or stale replica (works during partition, but stale)
5. **Read-your-own-writes**: a practical middle ground — writers read from primary, everyone else from replica
6. **Bank transfers** show why CP matters — wrong balances cause real financial harm

### Next Up

In **Notebook 3**, we'll zoom out and discuss **choosing the right database for your use case** — which databases are CP, which are AP, and how to decide in a system design interview.